# 07_Customer Analytics

## 1. Objectifs de l'analyse

Ce notebook étudie les caractéristiques démographiques, socio-économiques et les comportements d'achat des clients présents dans `DimCustomer`.

Les profils sont enrichis avec les ventes Internet observées entre 2014 et 2016. L'analyse distingue la population totale de la dimension client et les clients ayant effectué au moins un achat dans la table de faits.

In [2]:
# Importer les bibliothèques nécessaires
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Définir le chemin du projet
BASE_DIR = Path.cwd().parent
PROCESSED_DIR = BASE_DIR / "data" / "processed"

# Charger les cinq tables préparées
dim_date = pd.read_csv(
    PROCESSED_DIR / "DimDate.csv",
    parse_dates=["Date"]
)

dim_customer = pd.read_csv(
    PROCESSED_DIR / "DimCustomer.csv",
    parse_dates=["BirthDate", "DateFirstPurchase"]
)

dim_product = pd.read_csv(
    PROCESSED_DIR / "DimProduct.csv",
    parse_dates=["StartDate"]
)

fact_sales = pd.read_csv(
    PROCESSED_DIR / "FactInternetSales.csv",
    parse_dates=["OrderDate", "ShipDate"]
)

dim_territory = pd.read_csv(
    PROCESSED_DIR / "DimSalesTerritory.csv"
)

print("Chargement des données préparées terminé.")

Chargement des données préparées terminé.


## 2. Préparation du profil client et calcul de l'âge

In [4]:
# Afficher les colonnes disponibles dans la dimension clients
print("Colonnes de dim_customer :")
print(dim_customer.columns.tolist())

# Afficher un aperçu des données
display(dim_customer.head())

Colonnes de dim_customer :
['CustomerKey', 'FirstName', 'LastName', 'FullName', 'BirthDate', 'MaritalStatus', 'Gender', 'YearlyIncome', 'TotalChildren', 'NumberChildrenAtHome', 'Education', 'Occupation', 'HouseOwnerFlag', 'NumberCarsOwned', 'AddressLine1', 'DateFirstPurchase', 'CommuteDistance']


,CustomerKey,FirstName,LastName,FullName,BirthDate,MaritalStatus,Gender,YearlyIncome,TotalChildren,NumberChildrenAtHome,Education,Occupation,HouseOwnerFlag,NumberCarsOwned,AddressLine1,DateFirstPurchase,CommuteDistance
0,11000,Jon,Yang,"Yang, Jon",1966-04-08,M,M,90000,2,0,Bachelors,Professional,1,0,3761 N. 14th St,2014-01-22,1-2 Miles
1,11001,Eugene,Huang,"Huang, Eugene",1965-05-14,S,M,60000,3,3,Bachelors,Professional,0,1,2243 W St.,2014-01-18,0-1 Miles
2,11002,Ruben,Torres,"Torres, Ruben",1965-08-12,M,M,60000,3,3,Bachelors,Professional,1,1,5844 Linden Land,2014-01-10,2-5 Miles
3,11003,Christy,Zhu,"Zhu, Christy",1968-02-15,S,F,70000,0,0,Bachelors,Professional,0,1,1825 Village Pl.,2014-01-01,5-10 Miles
4,11004,Elizabeth,Johnson,"Johnson, Elizabeth",1968-08-08,S,F,80000,5,5,Bachelors,Professional,1,4,7553 Harness Circle,2014-01-26,1-2 Miles


In [5]:
# Créer une copie de la dimension clients
customer_analysis = dim_customer.copy()

# Définir la date de référence de l'analyse
date_reference = fact_sales["OrderDate"].max()

# Calculer l'âge des clients à la date de référence
customer_analysis["Age"] = (
    date_reference.year
    - customer_analysis["BirthDate"].dt.year
    - (
        (
            customer_analysis["BirthDate"].dt.month * 100
            + customer_analysis["BirthDate"].dt.day
        )
        > (date_reference.month * 100 + date_reference.day)
    ).astype("int")
)

# Examiner la distribution des âges
print("Date de référence :", date_reference.date())
print("\nValeurs manquantes dans Age :",
      customer_analysis["Age"].isna().sum())

display(customer_analysis["Age"].describe().round(2))

# Afficher les premières lignes
display(
    customer_analysis[
        ["CustomerKey", "FullName", "BirthDate", "Age"]
    ].head()
)

Date de référence : 2016-12-30

Valeurs manquantes dans Age : 0


count    18484.00
mean        54.42
std         11.25
min         36.00
25%         46.00
50%         53.00
75%         62.00
max        106.00
Name: Age, dtype: float64

,CustomerKey,FullName,BirthDate,Age
0,11000,"Yang, Jon",1966-04-08,50
1,11001,"Huang, Eugene",1965-05-14,51
2,11002,"Torres, Ruben",1965-08-12,51
3,11003,"Zhu, Christy",1968-02-15,48
4,11004,"Johnson, Elizabeth",1968-08-08,48


In [6]:
# Définir les tranches d'âge
bornes_age = [0, 44, 54, 64, 74, np.inf]

labels_age = [
    "36–44 ans",
    "45–54 ans",
    "55–64 ans",
    "65–74 ans",
    "75 ans et plus"
]

customer_analysis["AgeGroup"] = pd.cut(
    customer_analysis["Age"],
    bins=bornes_age,
    labels=labels_age,
    right=True
)

# Calculer les effectifs et les proportions
repartition_age = (
    customer_analysis["AgeGroup"]
    .value_counts(sort=False)
    .rename("Nombre_Clients")
    .reset_index()
)

repartition_age["Pourcentage_%"] = (
    repartition_age["Nombre_Clients"]
    / len(customer_analysis) * 100
)

display(repartition_age.round(2))

,AgeGroup,Nombre_Clients,Pourcentage_%
0,36–44 ans,3880,20.99
1,45–54 ans,6335,34.27
2,55–64 ans,4672,25.28
3,65–74 ans,2586,13.99
4,75 ans et plus,1011,5.47


In [7]:
# Définir les variables démographiques à analyser
variables_demographiques = [
    "Gender",
    "MaritalStatus",
    "Education",
    "Occupation"
]

# Calculer les effectifs et les pourcentages
for variable in variables_demographiques:

    distribution = (
        customer_analysis[variable]
        .value_counts(dropna=False)
        .rename("Nombre_Clients")
        .to_frame()
    )

    distribution["Pourcentage_%"] = (
        distribution["Nombre_Clients"]
        / len(customer_analysis) * 100
    ).round(2)

    print(f"\nRépartition selon : {variable}")

    display(distribution)


Répartition selon : Gender


,Nombre_Clients,Pourcentage_%
Gender,,
M,9351,50.59
F,9133,49.41



Répartition selon : MaritalStatus


,Nombre_Clients,Pourcentage_%
MaritalStatus,,
M,10011,54.16
S,8473,45.84



Répartition selon : Education


,Nombre_Clients,Pourcentage_%
Education,,
Bachelors,5356,28.98
Partial College,5064,27.40
High School,3294,17.82
Graduate Degree,3189,17.25
Partial High School,1581,8.55



Répartition selon : Occupation


,Nombre_Clients,Pourcentage_%
Occupation,,
Professional,5520,29.86
Skilled Manual,4577,24.76
Management,3075,16.64
Clerical,2928,15.84
Manual,2384,12.90


In [8]:
# Définir les variables numériques liées au profil socio-économique
variables_socio_economiques = [
    "YearlyIncome",
    "TotalChildren",
    "NumberChildrenAtHome",
    "NumberCarsOwned"
]

# Examiner les statistiques descriptives
display(
    customer_analysis[variables_socio_economiques]
    .describe()
    .round(2)
)

# Examiner la distribution des catégories de déplacement
distribution_deplacement = (
    customer_analysis["CommuteDistance"]
    .value_counts()
    .to_frame("Nombre_Clients")
)

distribution_deplacement["Pourcentage_%"] = (
    distribution_deplacement["Nombre_Clients"]
    / len(customer_analysis) * 100
).round(2)

display(distribution_deplacement)

,YearlyIncome,TotalChildren,NumberChildrenAtHome,NumberCarsOwned
count,18484.00,18484.00,18484.00,18484.00
mean,57305.78,1.84,1.00,1.50
std,32285.84,1.61,1.52,1.14
min,10000.00,0.00,0.00,0.00
25%,30000.00,0.00,0.00,1.00
50%,60000.00,2.00,0.00,2.00
75%,70000.00,3.00,2.00,2.00
max,170000.00,5.00,5.00,4.00


,Nombre_Clients,Pourcentage_%
CommuteDistance,,
0-1 Miles,6310,34.14
2-5 Miles,3234,17.50
1-2 Miles,3232,17.49
5-10 Miles,3214,17.39
10+ Miles,2494,13.49


## 3. Enrichissement des profils avec les données de ventes

In [10]:
# Agréger les ventes au niveau de chaque client
ventes_par_client = (
    fact_sales
    .groupby("CustomerKey")
    .agg(
        Chiffre_Affaires=("SalesAmount", "sum"),
        Cout_Total_Estime=("CalculatedTotalCost", "sum"),
        Nombre_Commandes=("SalesOrderNumber", "nunique"),
        Nombre_Lignes=("SalesOrderNumber", "count"),
        Quantite_Achetee=("OrderQuantity", "sum"),
        Nombre_Produits=("ProductKey", "nunique"),
        Date_Premier_Achat=("OrderDate", "min"),
        Date_Dernier_Achat=("OrderDate", "max")
    )
    .reset_index()
)

# Calculer le bénéfice brut estimé par client
ventes_par_client["Benefice_Brut_Estime"] = (
    ventes_par_client["Chiffre_Affaires"]
    - ventes_par_client["Cout_Total_Estime"]
)

# Calculer le panier moyen par client
ventes_par_client["Panier_Moyen"] = (
    ventes_par_client["Chiffre_Affaires"]
    / ventes_par_client["Nombre_Commandes"]
)

# Vérifier les résultats
print("Nombre de clients ayant effectué des achats :", len(ventes_par_client))
print("Nombre de commandes :", ventes_par_client["Nombre_Commandes"].sum())

display(ventes_par_client.head())

Nombre de clients ayant effectué des achats : 17918
Nombre de commandes : 26683


,CustomerKey,Chiffre_Affaires,Cout_Total_Estime,Nombre_Commandes,Nombre_Lignes,Quantite_Achetee,Nombre_Produits,Date_Premier_Achat,Date_Dernier_Achat,Benefice_Brut_Estime,Panier_Moyen
0,11000,8248.99,6647.4539,3,8,9,8,2014-01-22,2016-05-04,1601.5361,2749.663333
1,11001,6383.88,4915.9333,3,11,17,10,2014-01-18,2016-12-12,1467.9467,2127.960000
2,11002,8114.04,4659.1599,3,4,4,4,2014-01-10,2016-02-27,3454.8801,2704.680000
3,11003,8139.29,6584.3180,3,9,10,9,2014-01-01,2016-05-11,1554.9720,2713.096667
4,11004,8196.01,6189.1291,3,6,8,6,2014-01-26,2016-05-02,2006.8809,2732.003333


In [11]:
# Fusionner les profils clients avec leurs indicateurs de vente
customer_sales = customer_analysis.merge(
    ventes_par_client,
    on="CustomerKey",
    how="left",
    validate="one_to_one"
)

# Identifier les clients sans vente Internet dans la période
clients_sans_ventes = customer_sales["Chiffre_Affaires"].isna().sum()

# Remplacer les valeurs manquantes des indicateurs quantitatifs par zéro
colonnes_ventes = [
    "Chiffre_Affaires",
    "Cout_Total_Estime",
    "Nombre_Commandes",
    "Nombre_Lignes",
    "Quantite_Achetee",
    "Nombre_Produits",
    "Benefice_Brut_Estime",
    "Panier_Moyen"
]

customer_sales[colonnes_ventes] = (
    customer_sales[colonnes_ventes].fillna(0)
)

# Vérifier la structure de la table enrichie
print("Nombre total de clients :", len(customer_sales))
print("Clients sans ventes Internet :", clients_sans_ventes)
print("Clients avec ventes Internet :",
      len(customer_sales) - clients_sans_ventes)

print("\nValeurs manquantes dans les indicateurs de vente :")
display(customer_sales[colonnes_ventes].isna().sum())

display(
    customer_sales[
        [
            "CustomerKey",
            "Age",
            "Gender",
            "YearlyIncome",
            "Chiffre_Affaires",
            "Nombre_Commandes",
            "Panier_Moyen"
        ]
    ].head()
)

Nombre total de clients : 18484
Clients sans ventes Internet : 566
Clients avec ventes Internet : 17918

Valeurs manquantes dans les indicateurs de vente :


Chiffre_Affaires        0
Cout_Total_Estime       0
Nombre_Commandes        0
Nombre_Lignes           0
Quantite_Achetee        0
Nombre_Produits         0
Benefice_Brut_Estime    0
Panier_Moyen            0
dtype: int64

,CustomerKey,Age,Gender,YearlyIncome,Chiffre_Affaires,Nombre_Commandes,Panier_Moyen
0,11000,50,M,90000,8248.99,3.0,2749.663333
1,11001,51,M,60000,6383.88,3.0,2127.960000
2,11002,51,M,60000,8114.04,3.0,2704.680000
3,11003,48,F,70000,8139.29,3.0,2713.096667
4,11004,48,F,80000,8196.01,3.0,2732.003333


## 4. Analyse des ventes par tranche d'âge

In [13]:
# Définir les tranches d'âge
bornes_age = [35, 44, 54, 64, 74, np.inf]

etiquettes_age = [
    "36–44 ans",
    "45–54 ans",
    "55–64 ans",
    "65–74 ans",
    "75 ans et plus"
]

customer_sales["AgeGroup"] = pd.cut(
    customer_sales["Age"],
    bins=bornes_age,
    labels=etiquettes_age,
    right=True
)

# Analyser les indicateurs commerciaux par tranche d'âge
performance_age = (
    customer_sales
    .groupby("AgeGroup", observed=False)
    .agg(
        Nombre_Clients=("CustomerKey", "nunique"),
        Clients_Acheteurs=("Nombre_Commandes",
                           lambda x: (x > 0).sum()),
        Chiffre_Affaires=("Chiffre_Affaires", "sum"),
        Nombre_Commandes=("Nombre_Commandes", "sum"),
        Benefice_Brut_Estime=("Benefice_Brut_Estime", "sum")
    )
    .reset_index()
)

# Calculer les indicateurs moyens
performance_age["Taux_Clients_Acheteurs_%"] = (
    performance_age["Clients_Acheteurs"]
    / performance_age["Nombre_Clients"] * 100
)

performance_age["CA_Moyen_Par_Acheteur"] = (
    performance_age["Chiffre_Affaires"]
    / performance_age["Clients_Acheteurs"]
)

performance_age["Panier_Moyen"] = (
    performance_age["Chiffre_Affaires"]
    / performance_age["Nombre_Commandes"]
)

performance_age["Benefice_Moyen_Par_Acheteur"] = (
    performance_age["Benefice_Brut_Estime"]
    / performance_age["Clients_Acheteurs"]
)

# Afficher les résultats
display(
    performance_age.round(2)
)

,AgeGroup,Nombre_Clients,Clients_Acheteurs,Chiffre_Affaires,Nombre_Commandes,Benefice_Brut_Estime,Taux_Clients_Acheteurs_%,CA_Moyen_Par_Acheteur,Panier_Moyen,Benefice_Moyen_Par_Acheteur
0,36–44 ans,3880,3757,5422354.00,5463.0,409652.44,96.83,1443.27,992.56,109.04
1,45–54 ans,6335,6190,11483557.40,9562.0,834759.81,97.71,1855.18,1200.96,134.86
2,55–64 ans,4672,4520,8092588.03,6745.0,683937.04,96.75,1790.40,1199.79,151.31
3,65–74 ans,2586,2479,3367187.40,3607.0,224442.20,95.86,1358.28,933.51,90.54
4,75 ans et plus,1011,972,942149.76,1306.0,81564.49,96.14,969.29,721.40,83.91


## 5. Analyse des revenus et des dépenses

In [15]:
# Examiner la distribution des revenus annuels
distribution_revenus = (
    customer_sales["YearlyIncome"]
    .value_counts()
    .sort_index()
    .rename_axis("Revenu_Annuel")
    .reset_index(name="Nombre_Clients")
)

distribution_revenus["Pourcentage_%"] = (
    distribution_revenus["Nombre_Clients"]
    / len(customer_sales) * 100
).round(2)

print("Nombre de niveaux de revenus distincts :",
      customer_sales["YearlyIncome"].nunique())

display(distribution_revenus)

Nombre de niveaux de revenus distincts : 16


,Revenu_Annuel,Nombre_Clients,Pourcentage_%
0,10000,1155,6.25
1,20000,1767,9.56
2,30000,2287,12.37
3,40000,2747,14.86
4,50000,670,3.62
5,60000,3127,16.92
6,70000,2349,12.71
7,80000,1342,7.26
8,90000,842,4.56
9,100000,571,3.09


In [16]:
# Définir les tranches de revenus annuels
bornes_revenus = [
    0, 30000, 50000, 70000, 100000, np.inf
]

etiquettes_revenus = [
    "Faibles",
    "Modestes",
    "Intermédiaires",
    "Élevés",
    "Très élevés"
]

customer_sales["IncomeGroup"] = pd.cut(
    customer_sales["YearlyIncome"],
    bins=bornes_revenus,
    labels=etiquettes_revenus,
    right=False
)

# Vérifier la répartition des clients par tranche
distribution_income = (
    customer_sales
    .groupby("IncomeGroup", observed=False)
    .agg(
        Nombre_Clients=("CustomerKey", "nunique"),
        Clients_Acheteurs=(
            "Nombre_Commandes",
            lambda x: (x > 0).sum()
        )
    )
    .reset_index()
)

distribution_income["Pourcentage_%"] = (
    distribution_income["Nombre_Clients"]
    / len(customer_sales) * 100
).round(2)

display(distribution_income)

,IncomeGroup,Nombre_Clients,Clients_Acheteurs,Pourcentage_%
0,Faibles,2922,2820,15.81
1,Modestes,5034,4878,27.23
2,Intermédiaires,3797,3683,20.54
3,Élevés,4533,4411,24.52
4,Très élevés,2198,2126,11.89


In [17]:
# Agréger les performances commerciales par tranche de revenus
performance_revenus = (
    customer_sales
    .groupby("IncomeGroup", observed=False)
    .agg(
        Nombre_Clients=("CustomerKey", "nunique"),
        Clients_Acheteurs=(
            "Nombre_Commandes",
            lambda x: (x > 0).sum()
        ),
        Chiffre_Affaires=("Chiffre_Affaires", "sum"),
        Nombre_Commandes=("Nombre_Commandes", "sum"),
        Benefice_Brut_Estime=("Benefice_Brut_Estime", "sum")
    )
    .reset_index()
)

# Calculer les indicateurs moyens
performance_revenus["Taux_Clients_Acheteurs_%"] = (
    performance_revenus["Clients_Acheteurs"]
    / performance_revenus["Nombre_Clients"] * 100
)

performance_revenus["CA_Moyen_Par_Acheteur"] = (
    performance_revenus["Chiffre_Affaires"]
    / performance_revenus["Clients_Acheteurs"]
)

performance_revenus["Panier_Moyen"] = (
    performance_revenus["Chiffre_Affaires"]
    / performance_revenus["Nombre_Commandes"]
)

performance_revenus["Benefice_Moyen_Par_Acheteur"] = (
    performance_revenus["Benefice_Brut_Estime"]
    / performance_revenus["Clients_Acheteurs"]
)

# Afficher les résultats
display(performance_revenus.round(2))

,IncomeGroup,Nombre_Clients,Clients_Acheteurs,Chiffre_Affaires,Nombre_Commandes,Benefice_Brut_Estime,Taux_Clients_Acheteurs_%,CA_Moyen_Par_Acheteur,Panier_Moyen,Benefice_Moyen_Par_Acheteur
0,Faibles,2922,2820,3400346.96,3835.0,333604.79,96.51,1205.80,886.66,118.30
1,Modestes,5034,4878,7134876.45,7054.0,460974.62,96.90,1462.66,1011.47,94.50
2,Intermédiaires,3797,3683,5026292.44,5126.0,325253.82,97.00,1364.73,980.55,88.31
3,Élevés,4533,4411,8928207.48,7159.0,736039.61,97.31,2024.08,1247.13,166.86
4,Très élevés,2198,2126,4818113.27,3509.0,378483.13,96.72,2266.28,1373.07,178.03


## 6. Analyse selon les caractéristiques démographiques et familiales

In [19]:
# Définir les variables catégorielles à analyser
variables_profil = ["Gender", "MaritalStatus"]

# Calculer les performances commerciales
for variable in variables_profil:

    performance = (
        customer_sales
        .groupby(variable)
        .agg(
            Nombre_Clients=("CustomerKey", "nunique"),
            Clients_Acheteurs=(
                "Nombre_Commandes",
                lambda x: (x > 0).sum()
            ),
            Chiffre_Affaires=("Chiffre_Affaires", "sum"),
            Nombre_Commandes=("Nombre_Commandes", "sum"),
            Benefice_Brut_Estime=("Benefice_Brut_Estime", "sum")
        )
        .reset_index()
    )

    performance["Taux_Clients_Acheteurs_%"] = (
        performance["Clients_Acheteurs"]
        / performance["Nombre_Clients"] * 100
    )

    performance["CA_Moyen_Par_Acheteur"] = (
        performance["Chiffre_Affaires"]
        / performance["Clients_Acheteurs"]
    )

    performance["Panier_Moyen"] = (
        performance["Chiffre_Affaires"]
        / performance["Nombre_Commandes"]
    )

    performance["Benefice_Moyen_Par_Acheteur"] = (
        performance["Benefice_Brut_Estime"]
        / performance["Clients_Acheteurs"]
    )

    print(f"\nPerformance selon : {variable}")
    display(performance.round(2))


Performance selon : Gender


,Gender,Nombre_Clients,Clients_Acheteurs,Chiffre_Affaires,Nombre_Commandes,Benefice_Brut_Estime,Taux_Clients_Acheteurs_%,CA_Moyen_Par_Acheteur,Panier_Moyen,Benefice_Moyen_Par_Acheteur
0,F,9133,8841,14787798.84,13244.0,1099909.72,96.80,1672.64,1116.57,124.41
1,M,9351,9077,14520037.76,13439.0,1134446.25,97.07,1599.65,1080.44,124.98



Performance selon : MaritalStatus


,MaritalStatus,Nombre_Clients,Clients_Acheteurs,Chiffre_Affaires,Nombre_Commandes,Benefice_Brut_Estime,Taux_Clients_Acheteurs_%,CA_Moyen_Par_Acheteur,Panier_Moyen,Benefice_Moyen_Par_Acheteur
0,M,10011,9708,15157640.36,14596.0,1093127.46,96.97,1561.36,1038.48,112.6
1,S,8473,8210,14150196.23,12087.0,1141228.52,96.90,1723.53,1170.70,139.0


In [20]:
# Définir les variables liées à la composition familiale
variables_familiales = [
    "TotalChildren",
    "NumberChildrenAtHome"
]

# Calculer les performances commerciales
for variable in variables_familiales:

    performance = (
        customer_sales
        .groupby(variable)
        .agg(
            Nombre_Clients=("CustomerKey", "nunique"),
            Clients_Acheteurs=(
                "Nombre_Commandes",
                lambda x: (x > 0).sum()
            ),
            Chiffre_Affaires=("Chiffre_Affaires", "sum"),
            Nombre_Commandes=("Nombre_Commandes", "sum"),
            Benefice_Brut_Estime=("Benefice_Brut_Estime", "sum")
        )
        .reset_index()
    )

    performance["Taux_Clients_Acheteurs_%"] = (
        performance["Clients_Acheteurs"]
        / performance["Nombre_Clients"] * 100
    )

    performance["CA_Moyen_Par_Acheteur"] = (
        performance["Chiffre_Affaires"]
        / performance["Clients_Acheteurs"]
    )

    performance["Panier_Moyen"] = (
        performance["Chiffre_Affaires"]
        / performance["Nombre_Commandes"]
    )

    performance["Benefice_Moyen_Par_Acheteur"] = (
        performance["Benefice_Brut_Estime"]
        / performance["Clients_Acheteurs"]
    )

    print(f"\nPerformance selon : {variable}")
    display(performance.round(2))


Performance selon : TotalChildren


,TotalChildren,Nombre_Clients,Clients_Acheteurs,Chiffre_Affaires,Nombre_Commandes,Benefice_Brut_Estime,Taux_Clients_Acheteurs_%,CA_Moyen_Par_Acheteur,Panier_Moyen,Benefice_Moyen_Par_Acheteur
0,0,5165,5033,8620890.37,7665.0,671400.51,97.44,1712.87,1124.71,133.40
1,1,3619,3526,6354426.05,5222.0,406094.89,97.43,1802.16,1216.86,115.17
2,2,3779,3666,6318784.75,5494.0,544479.84,97.01,1723.62,1150.12,148.52
3,3,2194,2121,3384289.34,3049.0,163120.73,96.67,1595.61,1109.97,76.91
4,4,2303,2217,2928966.56,3260.0,300932.65,96.27,1321.14,898.46,135.74
5,5,1424,1355,1700479.53,1993.0,148327.36,95.15,1254.97,853.23,109.47



Performance selon : NumberChildrenAtHome


,NumberChildrenAtHome,Nombre_Clients,Clients_Acheteurs,Chiffre_Affaires,Nombre_Commandes,Benefice_Brut_Estime,Taux_Clients_Acheteurs_%,CA_Moyen_Par_Acheteur,Panier_Moyen,Benefice_Moyen_Par_Acheteur
0,0,11116,10789,17898131.98,15876.0,1299126.18,97.06,1658.92,1127.37,120.41
1,1,2460,2381,3532656.69,3390.0,247104.88,96.79,1483.69,1042.08,103.78
2,2,1648,1607,2954006.39,2344.0,255117.68,97.51,1838.21,1260.24,158.75
3,3,1204,1157,1461794.31,1692.0,165999.90,96.10,1263.44,863.94,143.47
4,4,1089,1046,1589075.87,1786.0,142247.13,96.05,1519.19,889.74,135.99
5,5,967,938,1872171.35,1595.0,124760.20,97.00,1995.92,1173.78,133.01


## 7. Analyse selon le niveau d'études et la profession

In [22]:
# Définir les variables socio-professionnelles
variables_socio_pro = ["Education", "Occupation"]

# Analyser les performances commerciales
for variable in variables_socio_pro:

    performance = (
        customer_sales
        .groupby(variable)
        .agg(
            Nombre_Clients=("CustomerKey", "nunique"),
            Clients_Acheteurs=(
                "Nombre_Commandes",
                lambda x: (x > 0).sum()
            ),
            Chiffre_Affaires=("Chiffre_Affaires", "sum"),
            Nombre_Commandes=("Nombre_Commandes", "sum"),
            Benefice_Brut_Estime=("Benefice_Brut_Estime", "sum")
        )
        .reset_index()
    )

    # Calculer les indicateurs moyens
    performance["Taux_Clients_Acheteurs_%"] = (
        performance["Clients_Acheteurs"]
        / performance["Nombre_Clients"] * 100
    )

    performance["CA_Moyen_Par_Acheteur"] = (
        performance["Chiffre_Affaires"]
        / performance["Clients_Acheteurs"]
    )

    performance["Panier_Moyen"] = (
        performance["Chiffre_Affaires"]
        / performance["Nombre_Commandes"]
    )

    performance["Benefice_Moyen_Par_Acheteur"] = (
        performance["Benefice_Brut_Estime"]
        / performance["Clients_Acheteurs"]
    )

    print(f"\nPerformance selon : {variable}")
    display(performance.round(2))


Performance selon : Education


,Education,Nombre_Clients,Clients_Acheteurs,Chiffre_Affaires,Nombre_Commandes,Benefice_Brut_Estime,Taux_Clients_Acheteurs_%,CA_Moyen_Par_Acheteur,Panier_Moyen,Benefice_Moyen_Par_Acheteur
0,Bachelors,5356,5234,9887500.13,8167.0,711923.45,97.72,1889.09,1210.66,136.02
1,Graduate Degree,3189,3092,5451817.18,4666.0,397058.74,96.96,1763.20,1168.41,128.41
2,High School,3294,3173,4627720.87,4568.0,314518.49,96.33,1458.47,1013.07,99.12
3,Partial College,5064,4903,7708608.49,7234.0,678833.12,96.82,1572.22,1065.61,138.45
4,Partial High School,1581,1516,1632189.92,2048.0,132022.18,95.89,1076.64,796.97,87.09



Performance selon : Occupation


,Occupation,Nombre_Clients,Clients_Acheteurs,Chiffre_Affaires,Nombre_Commandes,Benefice_Brut_Estime,Taux_Clients_Acheteurs_%,CA_Moyen_Par_Acheteur,Panier_Moyen,Benefice_Moyen_Par_Acheteur
0,Clerical,2928,2851,4677596.82,4267.0,376241.62,97.37,1640.69,1096.23,131.97
1,Management,3075,2978,5457718.46,4636.0,336450.44,96.85,1832.68,1177.25,112.98
2,Manual,2384,2309,2853709.62,3085.0,258380.63,96.85,1235.91,925.03,111.90
3,Professional,5520,5361,9893592.37,8365.0,805571.34,97.12,1845.48,1182.74,150.27
4,Skilled Manual,4577,4419,6425219.31,6330.0,457711.96,96.55,1454.00,1015.04,103.58


## 8. Indicateurs globaux de la clientèle

In [24]:
# Construire les indicateurs globaux de la clientèle
synthese_clients = pd.DataFrame({
    "Indicateur": [
        "Nombre total de clients",
        "Nombre de clients acheteurs",
        "Taux de clients acheteurs (%)",
        "Chiffre d'affaires total",
        "Nombre de commandes",
        "Nombre total de lignes de vente",
        "Quantité totale achetée",
        "CA moyen par acheteur",
        "Panier moyen",
        "Bénéfice brut estimé",
        "Taux de marque estimé (%)"
    ],
    "Valeur": [
        customer_sales["CustomerKey"].nunique(),
        (customer_sales["Nombre_Commandes"] > 0).sum(),
        (customer_sales["Nombre_Commandes"] > 0).mean() * 100,
        customer_sales["Chiffre_Affaires"].sum(),
        customer_sales["Nombre_Commandes"].sum(),
        customer_sales["Nombre_Lignes"].sum(),
        customer_sales["Quantite_Achetee"].sum(),
        customer_sales.loc[
            customer_sales["Nombre_Commandes"] > 0,
            "Chiffre_Affaires"
        ].mean(),
        customer_sales["Chiffre_Affaires"].sum()
        / customer_sales["Nombre_Commandes"].sum(),
        customer_sales["Benefice_Brut_Estime"].sum(),
        customer_sales["Benefice_Brut_Estime"].sum()
        / customer_sales["Chiffre_Affaires"].sum() * 100
    ]
})

display(synthese_clients.round(2))

,Indicateur,Valeur
0,Nombre total de clients,18484.00
1,Nombre de clients acheteurs,17918.00
2,Taux de clients acheteurs (%),96.94
3,Chiffre d'affaires total,29307836.59
4,Nombre de commandes,26683.00
5,Nombre total de lignes de vente,58189.00
6,Quantité totale achetée,91321.00
7,CA moyen par acheteur,1635.66
8,Panier moyen,1098.37
9,Bénéfice brut estimé,2234355.98


### Synthèse de l'analyse du profil et du comportement des clients

L'analyse de la dimension client, composée de 18 484 individus, met en évidence une clientèle principalement âgée de 45 à 64 ans, qui représente environ 59,55 % de l'effectif total. La répartition entre hommes et femmes est équilibrée, tandis que les clients mariés sont légèrement majoritaires.

Sur le plan socio-économique, le revenu annuel médian s'établit à 60 000. Les catégories professionnelles Professional et Skilled Manual sont les plus représentées. Les niveaux d'études Bachelors et Partial College regroupent également une part importante de la clientèle.

L'enrichissement des profils clients avec les données de ventes permet d'identifier 17 918 acheteurs, soit 96,94 % de la dimension client. Ces clients ont généré un chiffre d'affaires total de 29 307 836,59 à travers 26 683 commandes et 58 189 lignes de vente.

L'analyse par tranche d'âge montre que les clients de 45 à 54 ans génèrent le chiffre d'affaires total et le chiffre d'affaires moyen par acheteur les plus élevés. Les clients de 55 à 64 ans présentent, quant à eux, le bénéfice brut estimé moyen par acheteur le plus important parmi les tranches d'âge.

L'analyse des revenus révèle une relation globalement positive entre le niveau de revenu et les dépenses moyennes. Les groupes à revenus élevés et très élevés présentent les paniers moyens les plus importants. Le groupe aux revenus très élevés affiche également le meilleur bénéfice brut estimé moyen par acheteur.

Les caractéristiques familiales et socio-professionnelles font apparaître d'autres différences. Les clients célibataires présentent des dépenses moyennes supérieures à celles des clients mariés. Les clients ayant un niveau Bachelors et ceux exerçant une profession Professional affichent également des performances commerciales intéressantes.

Dans l'ensemble, ces résultats montrent que l'âge, le revenu, la situation familiale, le niveau d'études et la profession constituent des variables pertinentes pour étudier les différences de comportement d'achat. Leur interprétation doit toutefois tenir compte des interactions possibles entre ces caractéristiques.

**Limites méthodologiques :**

- Les 566 clients absents de FactInternetSales ne sont pas nécessairement dépourvus d'achats sur les autres canaux de vente.
- Les bénéfices et taux de marque présentés sont estimés à partir des coûts standards et ne représentent pas nécessairement les résultats comptables réels.
- Les analyses réalisées sont descriptives et ne permettent pas d'établir des relations de causalité entre les caractéristiques des clients et leurs dépenses.

## 9. Limites méthodologiques

- Les 566 clients de la dimension qui ne figurent pas dans `FactInternetSales` ne sont pas nécessairement dépourvus d'achats sur d'autres canaux.
- Les indicateurs moyens par acheteur sont calculés sur la période cumulée 2014-2016.
- Les bénéfices moyens par acheteur reposent sur des coûts estimés à partir des coûts standards.
- Les analyses sont descriptives. Elles ne permettent pas d'établir une relation de causalité entre les caractéristiques des clients et leurs dépenses.